In [1]:
import csv
import psycopg
import simplejson
import sqlite3

def importuj_dane(sciezka_csv):
    print(f"Rozpoczynam proces importu z pliku: {sciezka_csv}")
    
    # Odczyt danych z CSV
    dane_z_csv = []
    with open(sciezka_csv, 'r', encoding='utf-8') as plik_csv:
        reader = csv.reader(plik_csv)
        next(reader) 
        for row in reader:
            dane_z_csv.append(row)
            
    print(f"Wczytano {len(dane_z_csv)} rekordów. Przystępuję do ładowania...\n")

    # ==========================================
    # 1. POSTGRESQL - UŻYCIE MECHANIZMU COPY
    # ==========================================
    try:
        with open("database_creds.json") as db_con_file:
            creds = simplejson.loads(db_con_file.read())
            
        conn_pg = psycopg.connect(
            host=creds['host_name'], user=creds['user_name'],
            dbname=creds['db_name'], password=creds['password'], port=creds['port_number']
        )
        cursor_pg = conn_pg.cursor()
        
        # CZYSZCZENIE TABELI PRZED IMPORTEM
        cursor_pg.execute("TRUNCATE TABLE Klienci CASCADE;")
        
        # COPY
        zapytanie_copy = "COPY Klienci (Imie, Nazwisko, PESEL, Nr_Prawa_Jazdy, Telefon, Email) FROM STDIN"
        with cursor_pg.copy(zapytanie_copy) as copy_operation:
            for row in dane_z_csv:
                copy_operation.write_row(row)
                
        conn_pg.commit()
        print("✅ [PostgreSQL] Zakończono sukcesem (tabela wyczyszczona i załadowana przez COPY).")
        
        cursor_pg.close()
        conn_pg.close()
    except Exception as e:
        print("❌ [PostgreSQL] Błąd:", e)

    # ==========================================
    # 2. SQLITE - UŻYCIE BATCH INSERT
    # ==========================================
    try:
        conn_sl = sqlite3.connect('wypozyczalnia.db')
        cursor_sl = conn_sl.cursor()
        
        # CZYSZCZENIE TABELI PRZED IMPORTEM
        cursor_sl.execute("DELETE FROM Klienci;")
        
        # BATCH INSERT
        zapytanie_insert = """
            INSERT INTO Klienci (Imie, Nazwisko, PESEL, Nr_Prawa_Jazdy, Telefon, Email) 
            VALUES (?, ?, ?, ?, ?, ?)
        """
        cursor_sl.executemany(zapytanie_insert, dane_z_csv)
        conn_sl.commit()
        
        print("✅ [SQLite] Zakończono sukcesem (tabela wyczyszczona i załadowana przez BATCH INSERT).")
        
        cursor_sl.close()
        conn_sl.close()
    except Exception as e:
        print("❌ [SQLite] Błąd:", e)

if __name__ == "__main__":
    importuj_dane('klienci.csv')

Rozpoczynam proces importu z pliku: klienci.csv
Wczytano 10 rekordów. Przystępuję do ładowania...

✅ [PostgreSQL] Zakończono sukcesem (tabela wyczyszczona i załadowana przez COPY).
✅ [SQLite] Zakończono sukcesem (tabela wyczyszczona i załadowana przez BATCH INSERT).
